# Evidencia de Aprendizaje 1 — Carga y creación de dataset

**Proyecto Integrado III — Analítica de Datos**
Profesor: Andrés Felipe Callejas — PREICA2602B020038
Estudiante: Jesús González

Este cuaderno coordina las dos clases de la actividad:

| Archivo | Clase | Responsabilidad |
| --- | --- | --- |
| `src/hf_loader.py` | `HFDatasetLoader` | Conecta con Hugging Face y devuelve un `DataFrame` |
| `src/dataset_summarizer.py` | `DatasetSummarizer` | Resume el `DataFrame` y lo exporta a `.txt` |

**Flujo:** carga del dataset → siete transformaciones → resumen en
`reports/resumen_dataset_transformado.txt`.

Cada transformación devuelve un `DataFrame` nuevo y se muestra su resultado; el
último de ellos es el que se entrega a la clase de resumen, de modo que el
archivo `.txt` describe los datos transformados y no los originales.

## 1. Preparación del entorno

Se agrega la carpeta del proyecto al `sys.path` para poder importar el paquete
`src`. La búsqueda sube por los directorios padres, así el cuaderno funciona
tanto si se abre desde `EA1/` como desde la raíz del repositorio.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = next(
    (
        candidata
        for carpeta in [Path.cwd(), *Path.cwd().parents]
        for candidata in (carpeta, carpeta / "EA1")
        if (candidata / "src" / "hf_loader.py").exists()
    ),
    None,
)
if RAIZ is None:
    raise RuntimeError("No se encontró la carpeta src/: abre el cuaderno desde EA1/ o desde la raíz del repositorio.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.config import PROJECT_DIR, TARGET
from src.dataset_summarizer import DatasetSummarizer
from src.hf_loader import HFDatasetLoader


def mostrar(frame: pd.DataFrame, titulo: str, filas: int = 5) -> pd.DataFrame:
    """Imprime las dimensiones y devuelve las primeras filas para visualizarlas."""
    print(f"{titulo} → {frame.shape[0]} filas × {frame.shape[1]} columnas")
    return frame.head(filas)


print(f"Proyecto : {RAIZ.name}")
print(f"Dataset  : {TARGET.dataset_id}")
print(f"Split    : {TARGET.split}")
print(f"Descripción: {TARGET.descripcion}")

Proyecto : EA1
Dataset  : scikit-learn/iris
Split    : train
Descripción: Medidas morfologicas de 150 flores de tres especies de Iris.


## 2. Carga del dataset desde Hugging Face

`HFDatasetLoader` recibe el identificador del dataset y el split, hace la
conexión dentro de la clase y devuelve un `DataFrame` de pandas. El
identificador no está escrito dentro de la clase: se pasa como argumento, así
la misma clase sirve para cualquier dataset del Hub.

In [2]:
loader = HFDatasetLoader(TARGET.dataset_id, split=TARGET.split)
df = loader.load()

print(loader.info)
mostrar(df, "Datos originales")

Origen     : Hugging Face · scikit-learn/iris
Split      : train
Dimensiones: 150 filas x 6 columnas
Columnas   : Id, SepalLengthCm, SepalWidthCm, PetalLengthCm, PetalWidthCm, Species
Datos originales → 150 filas × 6 columnas


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 3. Transformaciones aplicadas

Se aplican siete transformaciones distintas, en cadena: cada una recibe el
`DataFrame` de la anterior y devuelve uno nuevo, sin modificar el original.

| # | Transformación | Tipo de operación |
| --- | --- | --- |
| 1 | Renombrar columnas al español | `rename` |
| 2 | Normalizar la etiqueta de especie | manipulación de texto |
| 3 | Seleccionar columnas de análisis | selección |
| 4 | Crear columnas derivadas | cálculo de columnas |
| 5 | Filtrar por área de pétalo | filtrado de filas |
| 6 | Añadir la media de área por especie | agregación por grupo |
| 7 | Ordenar por área de pétalo | ordenamiento |

### 3.1. Renombrar las columnas al español

Los nombres originales vienen en inglés y abreviados; se traducen a nombres claros para el análisis.

In [3]:
RENOMBRE = {
    "Id": "id",
    "SepalLengthCm": "largo_sepalo_cm",
    "SepalWidthCm": "ancho_sepalo_cm",
    "PetalLengthCm": "largo_petalo_cm",
    "PetalWidthCm": "ancho_petalo_cm",
    "Species": "especie",
}
df1 = df.rename(columns=RENOMBRE)
mostrar(df1, "1) Columnas renombradas")

1) Columnas renombradas → 150 filas × 6 columnas


,id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### 3.2. Normalizar la etiqueta de especie

La columna `especie` trae el prefijo `Iris-` y mayúsculas mezcladas; se deja en minúsculas y sin el prefijo para poder agrupar y comparar sin ruido.

In [4]:
df2 = df1.assign(especie=df1["especie"].str.replace("Iris-", "", regex=False).str.lower())
mostrar(df2, "2) Especie normalizada")

2) Especie normalizada → 150 filas × 6 columnas


,id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,1,5.1,3.5,1.4,0.2,setosa
1,2,4.9,3.0,1.4,0.2,setosa
2,3,4.7,3.2,1.3,0.2,setosa
3,4,4.6,3.1,1.5,0.2,setosa
4,5,5.0,3.6,1.4,0.2,setosa


### 3.3. Seleccionar las columnas de análisis

El identificador `id` solo numera las filas y no aporta información sobre las flores, así que se descarta. Quedan las cuatro medidas y la especie.

In [5]:
COLUMNAS = [
    "largo_sepalo_cm",
    "ancho_sepalo_cm",
    "largo_petalo_cm",
    "ancho_petalo_cm",
    "especie",
]
df3 = df2[COLUMNAS].copy()
mostrar(df3, "3) Columnas seleccionadas")

3) Columnas seleccionadas → 150 filas × 5 columnas


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


### 3.4. Crear columnas derivadas

El área del pétalo resume largo y ancho en una sola medida, y la proporción largo/ancho describe la forma de la flor. Ambas se calculan a partir de las columnas existentes.

In [6]:
df4 = df3.assign(
    area_petalo_cm2=(df3["largo_petalo_cm"] * df3["ancho_petalo_cm"]).round(2),
    ratio_petalo=(df3["largo_petalo_cm"] / df3["ancho_petalo_cm"]).round(2),
)
mostrar(df4, "4) Columnas derivadas")

4) Columnas derivadas → 150 filas × 7 columnas


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2,ratio_petalo
0,5.1,3.5,1.4,0.2,setosa,0.28,7.0
1,4.9,3.0,1.4,0.2,setosa,0.28,7.0
2,4.7,3.2,1.3,0.2,setosa,0.26,6.5
3,4.6,3.1,1.5,0.2,setosa,0.30,7.5
4,5.0,3.6,1.4,0.2,setosa,0.28,7.0


### 3.5. Filtrar las flores con pétalo grande

Se conservan únicamente las flores cuya área de pétalo supera la mediana del conjunto. El umbral se calcula con los datos, no se escribe a mano.

In [7]:
umbral = df4["area_petalo_cm2"].median()
df5 = df4[df4["area_petalo_cm2"] > umbral].copy()

print(f"Umbral (mediana del área) = {umbral:.2f} cm²")
mostrar(df5, "5) Filas filtradas")

Umbral (mediana del área) = 5.62 cm²
5) Filas filtradas → 75 filas × 7 columnas


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2,ratio_petalo
50,7.0,3.2,4.7,1.4,versicolor,6.58,3.36
51,6.4,3.2,4.5,1.5,versicolor,6.75,3.00
52,6.9,3.1,4.9,1.5,versicolor,7.35,3.27
54,6.5,2.8,4.6,1.5,versicolor,6.90,3.07
55,5.7,2.8,4.5,1.3,versicolor,5.85,3.46


### 3.6. Añadir la media de área por especie

Cada fila recibe el área promedio de su propia especie. Es una agregación por grupo que conserva todas las filas, para poder comparar cada flor contra su grupo.

In [8]:
df6 = df5.assign(
    area_media_especie=df5.groupby("especie")["area_petalo_cm2"].transform("mean").round(2)
)
mostrar(df6, "6) Media por especie agregada")

6) Media por especie agregada → 75 filas × 8 columnas


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2,ratio_petalo,area_media_especie
50,7.0,3.2,4.7,1.4,versicolor,6.58,3.36,6.82
51,6.4,3.2,4.5,1.5,versicolor,6.75,3.00,6.82
52,6.9,3.1,4.9,1.5,versicolor,7.35,3.27,6.82
54,6.5,2.8,4.6,1.5,versicolor,6.90,3.07,6.82
55,5.7,2.8,4.5,1.3,versicolor,5.85,3.46,6.82


### 3.7. Ordenar por área de pétalo

El resultado final se ordena de mayor a menor área, para que las flores más grandes queden arriba.

In [9]:
df_transformado = df6.sort_values("area_petalo_cm2", ascending=False).reset_index(drop=True)
mostrar(df_transformado, "7) DataFrame transformado", filas=8)

7) DataFrame transformado → 75 filas × 8 columnas


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2,ratio_petalo,area_media_especie
0,7.7,2.6,6.9,2.3,virginica,15.87,3.00,11.3
1,7.2,3.6,6.1,2.5,virginica,15.25,2.44,11.3
2,6.3,3.3,6.0,2.5,virginica,15.00,2.40,11.3
3,7.7,3.8,6.7,2.2,virginica,14.74,3.05,11.3
4,6.7,3.3,5.7,2.5,virginica,14.25,2.28,11.3
5,7.7,3.0,6.1,2.3,virginica,14.03,2.65,11.3
6,7.6,3.0,6.6,2.1,virginica,13.86,3.14,11.3
7,6.8,3.2,5.9,2.3,virginica,13.57,2.57,11.3


## 4. Resumen del `DataFrame` transformado

Se pasa el resultado de las siete transformaciones a `DatasetSummarizer`. La
clase no se conecta a Hugging Face: recibe los datos ya cargados y describe
filas, columnas, tipos, valores nulos, duplicados y estadísticas. Además
escribe el mismo texto en `reports/resumen_dataset_transformado.txt`.

In [10]:
resumen = DatasetSummarizer(
    df_transformado,
    nombre="Iris transformado (EA1)",
    origen=f"Hugging Face · {TARGET.dataset_id} · split={TARGET.split}",
)

texto = resumen.to_text()
print(texto)

ruta = resumen.save()
print(f"Resumen guardado en: {ruta.relative_to(PROJECT_DIR)}")

                                  RESUMEN DEL DATASET
Nombre    : Iris transformado (EA1)
Generado  : 2026-10-21 19:16:00
Origen    : Hugging Face · scikit-learn/iris · split=train

1. DIMENSIONES
----------------------------------------------------------------------------------------
Filas      : 75
Columnas   : 8
Memoria    : 5.5 KB
Duplicados : 1 filas (1.33 %)

2. COLUMNAS Y TIPOS
----------------------------------------------------------------------------------------
#  columna             tipo     no_nulos  nulos  nulos_%  unicos  ejemplo
-  ------------------  -------  --------  -----  -------  ------  ---------
1  largo_sepalo_cm     float64        75      0      0.0      24  7.7
2  ancho_sepalo_cm     float64        75      0      0.0      14  2.6
3  largo_petalo_cm     float64        75      0      0.0      24  6.9
4  ancho_petalo_cm     float64        75      0      0.0      14  2.3
5  especie             str            75      0      0.0       2  virginica
6  area_petalo_cm

Resumen guardado en: reports\resumen_dataset_transformado.txt


## 5. Verificación de la ejecución

Comprobaciones automáticas de que cada transformación se aplicó y de que el archivo `.txt` quedó escrito.

In [11]:
comprobaciones = [
    ("El dataset original se cargó completo", df.shape == (150, 6)),
    ("Se renombraron las columnas", list(df1.columns) != list(df.columns)),
    ("La especie quedó sin prefijo", not df2["especie"].str.startswith("Iris-").any()),
    ("Se descartó la columna identificadora", "id" not in df3.columns),
    ("Se crearon dos columnas derivadas", {"area_petalo_cm2", "ratio_petalo"} <= set(df4.columns)),
    ("El filtro redujo las filas", 0 < len(df5) < len(df4)),
    ("Se agregó la media por especie", "area_media_especie" in df6.columns),
    ("El resultado quedó ordenado", df_transformado["area_petalo_cm2"].is_monotonic_decreasing),
    ("El DataFrame final no tiene nulos", not df_transformado.isna().any().any()),
    ("El resumen se escribió en disco", ruta.exists() and ruta.stat().st_size > 0),
]

tabla = pd.DataFrame(comprobaciones, columns=["comprobación", "resultado"])
tabla["resultado"] = tabla["resultado"].map({True: "OK", False: "FALLA"})
display(tabla)

assert all(resultado for _, resultado in comprobaciones), "Alguna comprobación falló"
print(f"{len(comprobaciones)} comprobaciones superadas · {ruta.name}: {ruta.stat().st_size} bytes")

,comprobación,resultado
0,El dataset original se cargó completo,OK
1,Se renombraron las columnas,OK
2,La especie quedó sin prefijo,OK
3,Se descartó la columna identificadora,OK
4,Se crearon dos columnas derivadas,OK
5,El filtro redujo las filas,OK
6,Se agregó la media por especie,OK
7,El resultado quedó ordenado,OK
8,El DataFrame final no tiene nulos,OK
9,El resumen se escribió en disco,OK


10 comprobaciones superadas · resumen_dataset_transformado.txt: 2638 bytes


## 6. Conclusiones

- La separación en dos clases deja el cuaderno como simple coordinador: la
  conexión con Hugging Face vive en `HFDatasetLoader` y la descripción de los
  datos en `DatasetSummarizer`.
- Las siete transformaciones se aplicaron sobre copias, sin modificar el
  `DataFrame` original: al final siguen disponibles los datos crudos y los
  transformados.
- El filtro por mediana deja fuera la mitad del conjunto y, con ella, todas las
  flores *setosa*: su pétalo es el más pequeño de las tres especies, así que el
  análisis final compara únicamente *versicolor* y *virginica*.
- Al descartar el identificador aparece **una fila duplicada** (1,33 %): son dos
  flores con las mismas cuatro medidas. Es un hallazgo de calidad de datos, no
  un error de la carga, y el resumen lo reporta porque se calcula sobre los
  datos transformados.
- El archivo `reports/resumen_dataset_transformado.txt` corresponde al
  `DataFrame` transformado: sus columnas están en español e incluyen las dos
  columnas derivadas, lo que permite verificar la transformación.
- Cambiar de dataset solo requiere editar `src/config.py`; ninguna de las dos
  clases tiene el identificador escrito dentro.